### Import Dependencies

In [1]:
import openai
import instructor
from pydantic import BaseModel, Field

from qdrant_client import QdrantClient

In [2]:
from dotenv import load_dotenv

load_dotenv("../../.env")

True

### RAG Pipeline

In [6]:
client = instructor.from_provider(
    "openai/gpt-5.4-nano",
    mode=instructor.Mode.RESPONSES_TOOLS
)

In [3]:
class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="Answer to the question")

In [4]:
qdrant_client = QdrantClient(url="http://localhost:6333")

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )

    return response.data[0].embedding


def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }


def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context


def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}    
"""

    return prompt


def generate_answer(prompt):

    response, raw_response = client.create_with_completion(
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning={"effort": "none"},
        response_model=RAGGenerationResponse
    )

    return response


def rag_pipeline(question, top_k=5):

    retrieved_context = retrieve_data(question, k=top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_answer = {
        "data_object": answer,
        "answer": answer.answer,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"]
    }

    return final_answer

In [7]:
output = rag_pipeline("Do you have any earphones?")

In [8]:
output

{'data_object': RAGGenerationResponse(answer='Yes—there are multiple earphone options available, including kids wired earphones, wired iPhone/3.5mm earbuds with mic, and wireless Bluetooth earbuds (including sports styles and a headband option).'),
 'answer': 'Yes—there are multiple earphone options available, including kids wired earphones, wired iPhone/3.5mm earbuds with mic, and wireless Bluetooth earbuds (including sports styles and a headband option).',
 'question': 'Do you have any earphones?',
 'retrieved_context_ids': ['B0BBF2VC6X',
  'B0CH8DRD6K',
  'B09X9838WY',
  'B0BRV544MV',
  'B0CFHWF326'],
 'retrieved_context': ['Volkano Ninja Kids Earphones for Boys with Carry Case and Keyring - 3.5MM Stereo Jack - Wired Earbuds with Safe Volume 85dB - Earbuds for Kids for School for Online Learning/Travel/Tablet - Red/Blue HEARING PROTECTION: Made not just to look cool, but to be safe too. These kids\' earbuds with case limit their sound intensity to 85db to protect your kid\'s ears fr

In [9]:
print(output["answer"])

Yes—there are multiple earphone options available, including kids wired earphones, wired iPhone/3.5mm earbuds with mic, and wireless Bluetooth earbuds (including sports styles and a headband option).


### RAG Pipeline with Grounding Context

In [10]:
class RAGUsedContext(BaseModel):
    id: str = Field(description="ID of the item used to answer the question")
    description: str = Field(description="Description of the item used to answer the question")

class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="Answer to the question")
    references: list[RAGUsedContext] = Field(description="List of items used to answer the question")

In [18]:
qdrant_client = QdrantClient(url="http://localhost:6333")

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )

    return response.data[0].embedding


def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }


def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context


def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- If you are describing multiple products, list them out as a list.

Context:
{preprocessed_context}

Question:
{question}    
"""

    return prompt


def generate_answer(prompt):

    response, raw_response = client.create_with_completion(
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning={"effort": "none"},
        response_model=RAGGenerationResponse
    )

    return response


def rag_pipeline(question, top_k=5):

    retrieved_context = retrieve_data(question, k=top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_answer = {
        "data_object": answer,
        "answer": answer.answer,
        "references": answer.references,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"]
    }

    return final_answer

In [12]:
output = rag_pipeline("Do you have any earphones?")

In [13]:
output

{'data_object': RAGGenerationResponse(answer='Yes—we have several earphones available, including:\n- Volkano Ninja Kids wired earphones (with carry case) \n- 2-pack wired iPhone/MFi earphones with mic & volume control \n- Jesebang wireless Bluetooth 5.2 earbuds with charging case (IP7)\n- Wekily Bluetooth 5.3 wireless earbuds (IPX7)\n- MUSICOZY Bluetooth 5.3 sports headband headphones (with ENC mic)', references=[RAGUsedContext(id='B0BBF2VC6X', description='Volkano Ninja Kids wired earphones with safe volume (85dB) and carry case'), RAGUsedContext(id='B0CH8DRD6K', description='2-pack iPhone wired headphones (3.5mm, Apple MFi certified) with mic and volume control'), RAGUsedContext(id='B09X9838WY', description='Jesebang wireless Bluetooth 5.2 earbuds with charging case, 40H playtime, IP7 waterproof'), RAGUsedContext(id='B0BRV544MV', description='Wekily Bluetooth 5.3 wireless earbuds with charging case, 40H cycle playtime, IPX7'), RAGUsedContext(id='B0CFHWF326', description='MUSICOZY Blu

In [14]:
print(output["answer"])

Yes—we have several earphones available, including:
- Volkano Ninja Kids wired earphones (with carry case) 
- 2-pack wired iPhone/MFi earphones with mic & volume control 
- Jesebang wireless Bluetooth 5.2 earbuds with charging case (IP7)
- Wekily Bluetooth 5.3 wireless earbuds (IPX7)
- MUSICOZY Bluetooth 5.3 sports headband headphones (with ENC mic)


In [19]:
output = rag_pipeline("Do you have any earphones?", top_k=10)

In [20]:
output

{'data_object': RAGGenerationResponse(answer='Yes—there are several earphone options available:\n- Volkano Ninja Kids Earphones (wired, 85dB safe volume) with carry case\n- 2 Pack iPhone Wired Headphones (3.5mm jack, MFi certified) with microphone and volume control\n- Jesebang Wireless Earbuds (Bluetooth 5.2, charging case, mic) \n- Wekily Bluetooth 5.3 Wireless Earbuds (40H playtime, waterproof, 4-mic clear call)\n- MUSICOZY Bluetooth 5.3 Sports Headband Headphones (ENC mic)\n- Pamu Wireless Earbuds with ANC/ENC (Bluetooth 5.2, charging case)\n- Kids Wireless Headphones (adjustable headband, 3.5mm jack + Bluetooth)\n- Bluet00th Sleep Headphones Headband (sleep mask headband with built-in speakers)\n- RUNAR Wireless Neckband Earphones (Bluetooth 5.0, for running)\n\nWhich type do you want—wired or wireless, and for kids or adults?', references=[RAGUsedContext(id='B0BBF2VC6X', description='Volkano Ninja Kids Earphones (wired, 3.5mm) with carry case; 85dB safe volume.'), RAGUsedContext(

In [21]:
print(output["answer"])

Yes—there are several earphone options available:
- Volkano Ninja Kids Earphones (wired, 85dB safe volume) with carry case
- 2 Pack iPhone Wired Headphones (3.5mm jack, MFi certified) with microphone and volume control
- Jesebang Wireless Earbuds (Bluetooth 5.2, charging case, mic) 
- Wekily Bluetooth 5.3 Wireless Earbuds (40H playtime, waterproof, 4-mic clear call)
- MUSICOZY Bluetooth 5.3 Sports Headband Headphones (ENC mic)
- Pamu Wireless Earbuds with ANC/ENC (Bluetooth 5.2, charging case)
- Kids Wireless Headphones (adjustable headband, 3.5mm jack + Bluetooth)
- Bluet00th Sleep Headphones Headband (sleep mask headband with built-in speakers)
- RUNAR Wireless Neckband Earphones (Bluetooth 5.0, for running)

Which type do you want—wired or wireless, and for kids or adults?


In [22]:
output = rag_pipeline("Do you have any earphones?", top_k=15)

In [23]:
print(output["answer"])

Yes—there are earphone options available. Here are a few:

- Volkano Ninja Kids Earphones (wired, 3.5mm) with carry case and keyring
- 2 Pack iPhone Wired Headphones (MFi certified, 3.5mm) with microphone and volume control
- Jesebang Wireless Earbuds (Bluetooth 5.2) with charging case
- Wekily Bluetooth 5.3 Wireless Earbuds (Bluetooth earbuds with charging case)
- pamu Wireless ANC Earbuds (Bluetooth 5.2) with charging case
- Kids Bluetooth/3.5mm Jack Over-Ear Headphones with microphone (yellow)

If you tell me whether you want wired or wireless (and for which device), I can narrow it down.


In [24]:
output

{'data_object': RAGGenerationResponse(answer='Yes—there are earphone options available. Here are a few:\n\n- Volkano Ninja Kids Earphones (wired, 3.5mm) with carry case and keyring\n- 2 Pack iPhone Wired Headphones (MFi certified, 3.5mm) with microphone and volume control\n- Jesebang Wireless Earbuds (Bluetooth 5.2) with charging case\n- Wekily Bluetooth 5.3 Wireless Earbuds (Bluetooth earbuds with charging case)\n- pamu Wireless ANC Earbuds (Bluetooth 5.2) with charging case\n- Kids Bluetooth/3.5mm Jack Over-Ear Headphones with microphone (yellow)\n\nIf you tell me whether you want wired or wireless (and for which device), I can narrow it down.', references=[RAGUsedContext(id='B0BBF2VC6X', description='Volkano Ninja Kids Earphones (wired, 3.5mm) with safe volume and carry case'), RAGUsedContext(id='B0CH8DRD6K', description='2 Pack iPhone Wired Headphones (MFi certified, 3.5mm) with microphone and volume control'), RAGUsedContext(id='B09X9838WY', description='Jesebang Wireless Earbud (